In [1]:
# Sam Brown
# sam_brown@mines.edu
# June 26 2025
# Goal: Use new inter event form factor calculations to predict time since for events. Not entirely practical but useful for understanding of data

# Directory
import sys
sys.path.append("/Users/sambrown04/Documents/SURF/whillans-surf/notebooks/SURF")

# Imports
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

df = pd.read_csv("/Users/sambrown04/Documents/SURF/Preproc_data/10-18.csv")
df = df[508:3000] # Avoid "dark spots" for now

In [3]:
# Features and target
X = df[['tide_deriv', 'slip_size_standardized', 'high_t_evt', 'tide_height', 'A_diurn', 'A_semidiurn']]
y = df['time_since']

# Split
X_train, X_test, y_train, y_test = train_test_split(X, y , test_size = .2, random_state = 42)

# Standardize
x_scaler = StandardScaler()
X_train_scaled = x_scaler.fit_transform(X_train)
X_test_scaled = x_scaler.transform(X_test)

y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = y_scaler.transform(y_test.values.reshape(-1, 1))

# Pytorch tensors 
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_scaled, dtype=torch.float32)

In [5]:
# Neural Net
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.fc1 = nn.Linear(6, 24)
        self.fc2 = nn.Linear(24, 12)
        self.fc3 = nn.Linear(12,1)

    def forward(self,x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x) # Regressing
        return x

In [50]:
model = Net()
criterion = nn.MSELoss() # Loss for regression
optimizer = optim.Adam(model.parameters(), lr = .01)


# Training loop
epochs = 400
for epoch in range(epochs):
    model.train()
    optimizer.zero_grad() # Clears grad

    # Predictions and loss
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    # Backprop
    loss.backward()

    # Update params
    optimizer.step()
    
    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [20/400], Loss: 0.8098
Epoch [40/400], Loss: 0.7994
Epoch [60/400], Loss: 0.7930
Epoch [80/400], Loss: 0.7836
Epoch [100/400], Loss: 0.7661
Epoch [120/400], Loss: 0.7328
Epoch [140/400], Loss: 0.6726
Epoch [160/400], Loss: 0.5759
Epoch [180/400], Loss: 0.4843
Epoch [200/400], Loss: 0.3960
Epoch [220/400], Loss: 0.3308
Epoch [240/400], Loss: 0.2784
Epoch [260/400], Loss: 0.2435
Epoch [280/400], Loss: 0.2193
Epoch [300/400], Loss: 0.2077
Epoch [320/400], Loss: 0.2010
Epoch [340/400], Loss: 0.1945
Epoch [360/400], Loss: 0.1899
Epoch [380/400], Loss: 0.1856
Epoch [400/400], Loss: 0.1822


In [52]:
model.eval()
with torch.no_grad():
    y_pred_scaled = model(X_test_tensor)
    y_pred = y_scaler.inverse_transform(y_pred_scaled.numpy())
    y_test_orig = y_scaler.inverse_transform(y_test_tensor.numpy())

r2 = r2_score(y_test_orig, y_pred)
print(r2)

0.4580742120742798
